In [ ]:
import sys
import numpy as np
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
from importlib import reload
from scipy.stats import pearsonr, spearmanr
from sklearn.feature_selection import mutual_info_regression
from sklearn.preprocessing import StandardScaler
import statsmodels.api as sm
from suncalc import get_position, get_times
from datetime import datetime, timezone
from matplotlib.ticker import ScalarFormatter

In [2]:
sys.path.append("../../src")

In [3]:
import vis
import main

# If code has not been run before

In [46]:
temp = pd.read_csv('../../data/raw_data/ta_vp_rh_60_min_2022_09_01_2024_08_31_gap_filled_new.csv')
temp['datetime']=pd.to_datetime(temp['datetime'])
temp['datetime_UTC'] = temp['datetime']
temp['value'] = temp['ta']
temp = temp[temp['type'] == 'measured']

In [ ]:
temp

### Define nighttime

In [48]:
times = {}
lon = 7.85222
lat = 47.9959

for i in temp['datetime_UTC'].unique():
    times[i] = get_times(i, lon, lat)

temp['sunrise'] = temp['datetime_UTC'].apply(lambda x: times[x]['sunrise'].replace(tzinfo=timezone.utc))
temp['sunset'] = temp['datetime_UTC'].apply(lambda x: times[x]['sunset'].replace(tzinfo=timezone.utc))

temp['is_night'] = (temp['datetime_UTC'] < temp['sunrise']) | (temp['datetime_UTC'] > temp['sunset'])

In [49]:
temp = temp[temp['is_night']]

In [ ]:
temp

In [51]:
dates = temp['datetime_UTC'].dt.date.unique()

In [ ]:
len(dates)

## Heat island hours defined as hours with standard deviation greater than 2 and no precipitation on that day or day before
### No precipitation on that day or day before

In [39]:
prec23 = pd.read_csv('../../data/raw_data/FRCHEM_2023_Precipitation_Daily_UTC.csv')
prec22 = pd.read_csv('../../data/raw_data/FRCHEM_2022_Precipitation_Daily_UTC.csv')
prec24 = pd.read_csv('../../data/raw_data/FRCHEM_2024_Precipitation_Daily_UTC.csv')
# concat prec22 and prec23
prec = pd.concat([prec22, prec23, prec24])
prec['datetime_UTC'] = pd.to_datetime(prec['YYYYMMDDHHMM_From'], format='%Y%m%d%H%M', utc=True)
prec['Precipitation_Sum_mm_day_before'] = prec['Precipitation_Sum_mm'].shift(1)
no_prec = prec[(prec['datetime_UTC'].dt.date.isin(temp['datetime_UTC'].dt.date)) & (prec['Precipitation_Sum_mm'] == 0) & (prec['Precipitation_Sum_mm_day_before'] == 0)]['datetime_UTC'].dt.date
temp = temp[temp['datetime_UTC'].dt.date.isin(no_prec)]

In [40]:
no_prec = prec[(prec['datetime_UTC'].dt.date.isin(temp['datetime_UTC'].dt.date)) & (prec['Precipitation_Sum_mm'] == 0) & (prec['Precipitation_Sum_mm_day_before'] == 0)]['datetime_UTC'].dt.date
#heat_island_dates = heat_island_dates[heat_island_dates.dt.date.isin(no_prec)]

In [41]:
temp = temp[temp['datetime_UTC'].dt.date.isin(no_prec)]

### Define months and pivot

In [ ]:
temp_w = temp[temp['datetime_UTC'].dt.month.isin([12,1,2])]
temp_sp = temp[temp['datetime_UTC'].dt.month.isin([3,4,5])]
temp_s = temp[temp['datetime_UTC'].dt.month.isin([6,7,8])]
temp_a = temp[temp['datetime_UTC'].dt.month.isin([9,10,11])]

temp_w['datetime_UTC'] = temp_w['datetime_UTC'].astype(str)
temp_sp['datetime_UTC'] = temp_sp['datetime_UTC'].astype(str)
temp_s['datetime_UTC'] = temp_s['datetime_UTC'].astype(str)
temp_a['datetime_UTC'] = temp_a['datetime_UTC'].astype(str)
temp['datetime_UTC'] = temp['datetime_UTC'].astype(str)

temp_w = temp_w.pivot(index='station_id', columns='datetime_UTC', values='value')
temp_sp = temp_sp.pivot(index='station_id', columns='datetime_UTC', values='value')
temp_s = temp_s.pivot(index='station_id', columns='datetime_UTC', values='value')
temp_a = temp_a.pivot(index='station_id', columns='datetime_UTC', values='value')
temp = temp.pivot(index='station_id', columns='datetime_UTC', values='value')


### Standard deviation greater than 2

In [43]:
hin = temp.loc[:, temp.std() > 2].columns.values
hiwn = temp_w.loc[:, temp_w.std() > 2].columns.values
hisn = temp_s.loc[:, temp_s.std() > 2].columns.values
hian = temp_a.loc[:, temp_a.std() > 2].columns.values
hispn = temp_sp.loc[:, temp_sp.std() > 2].columns.values

In [ ]:
# count independent nights in hin list including after 6pm and before 6am of following day as one night
independent_nights_6pm = set()
for date in hin:
    date = pd.to_datetime(date)
    if date.hour >= 12:
        night_date = date.date()
    else:
        night_date = (date - pd.Timedelta(days=1)).date()
    independent_nights_6pm.add(night_date)
print(f"Number of independent nights in hin list: {len(independent_nights_6pm)}")

In [24]:
pd.DataFrame(hin).to_csv('../../data/processed_data/2_year_heat_island_dates_night.csv', index=False)

### Heat island statistics for all parameters

In [11]:
radius = 300
vars = gpd.read_parquet(f'../../data/processed_data/processed_station_params_{radius}.parquet')
vars_merged = gpd.read_parquet(f'../../data/processed_data/processed_station_params_merged_blocks_{radius}.parquet')
vars.index = vars['station_id']
vars_merged.index = vars_merged['station_id']
to_remove = ['station_id','station_no','station_name','station_long_name','station_type','station_lat','station_lon','station_elevation','mounting_structure','sky_view_factor','dominant_land_use','local_climate_zone','urban_atlas_class','urban_atlas_code','geometry','SVF_3D','station_elevation_diff']
vars = vars.drop(to_remove, axis=1)
vars['BuAre_sum'] = vars['BuAre_sum'] / (300**2*np.pi)
vars['BuEWA_3D_sum'] = vars['BuEWA_3D_sum'] / (300**2*np.pi)
vars["BuVol_3D_sum"] = vars["BuVol_3D_sum"] / (300**2*np.pi)

vars['HW_del_merged'] = vars['BuHt_wmean'] / vars_merged['BuW_delaunay_mean']
vars['HW_knn1_merged'] = vars['BuHt_wmean'] / vars_merged['BuW_knn1_mean']
vars['HW_knn5_merged'] = vars['BuHt_wmean'] / vars_merged['BuW_knn5_mean']

vars['BuW_delaunay_mean_merged'] = vars_merged['BuW_delaunay_mean']
vars['BuW_knn5_mean_merged'] = vars_merged['BuW_knn5_mean']

vars['BuERI_mode_merged'] = vars_merged['BuERI_mode']
vars['BuERI_wmean_merged'] = vars_merged['BuERI_wmean']

vars['BuDen_merged'] = vars_merged['BuDen']

temp = pd.read_csv('../../data/raw_data/ta_vp_rh_60_min_2022_09_01_2024_08_31_gap_filled_new.csv')
temp['datetime']=pd.to_datetime(temp['datetime'])
temp['datetime_UTC'] = temp['datetime']
temp['value'] = temp['ta']
temp = temp[temp['type'] == 'measured']

temp['datetime_UTC'] = temp['datetime_UTC'].astype(str)
temp = temp.pivot(index='station_id', columns='datetime_UTC', values='value')
temp.index = temp.index.str[2:]

In [ ]:
vars[['BuAre_sum','BuNum','BuDen','BuDen_merged']]

In [ ]:
stats_dict = {}
param_list = []
spearman_r_list = []
spearman_p_list = []
pearson_list = []
r_squared_list = []
rmse_list = []
cooks_d_list = []
mutual_info_list = []

for var in vars.columns:
    print(var)
    try:
        data, mean, std, spearman_corr, p_value, pearson_corr, r_squared, rmse, cooks_d, mi, y_pred, slope, intercept, bse, summary = main.stats_multiple_times(vars, var, hin, temp)
        param_list.append(var)
        spearman_r_list.append(spearman_corr)
        spearman_p_list.append(p_value)
        pearson_list.append(pearson_corr)
        r_squared_list.append(r_squared)
        rmse_list.append(rmse)
        cooks_d_list.append(cooks_d)
        mutual_info_list.append(mi)
    except UnboundLocalError as e:
        print(f"Error processing variable '{var}': {e}")
        continue
    
stats_dict['parameter'] = param_list
stats_dict['spearman_r'] = spearman_r_list
stats_dict['spearman_p'] = spearman_p_list
stats_dict['pearson_r'] = pearson_list
stats_dict['r_squared'] = r_squared_list
stats_dict['rmse'] = rmse_list
stats_dict['cooks_d'] = cooks_d_list
stats_dict['mutual_info'] = mutual_info_list


stats_df = pd.DataFrame(stats_dict)

In [57]:
# all params

In [29]:
stats_df.to_csv(f'../../data/processed_data/heat_island_stats_{radius}_.csv')
stats_df.to_excel(f'../../data/processed_data/heat_island_stats_{radius}_.xlsx', index=True, header=True)

# If code has been run before

In [4]:
radius = 300
vars = gpd.read_parquet(f'../../data/processed_data/processed_station_params_{radius}.parquet')
vars_merged = gpd.read_parquet(f'../../data/processed_data/processed_station_params_merged_blocks_{radius}.parquet')
vars.index = vars['station_id']
vars_merged.index = vars_merged['station_id']
to_remove = ['station_id','station_no','station_name','station_long_name','station_type','station_lat','station_lon','station_elevation','mounting_structure','sky_view_factor','dominant_land_use','local_climate_zone','urban_atlas_class','urban_atlas_code','geometry','SVF_3D','station_elevation_diff']
vars = vars.drop(to_remove, axis=1)
vars['BuAre_sum'] = vars['BuAre_sum'] / (300**2*np.pi)
vars['BuEWA_3D_sum'] = vars['BuEWA_3D_sum'] / (300**2*np.pi)
vars["BuVol_3D_sum"] = vars["BuVol_3D_sum"] / (300**2*np.pi)

vars['HW_del_merged'] = vars['BuHt_wmean'] / vars_merged['BuW_delaunay_mean']
vars['HW_knn1_merged'] = vars['BuHt_wmean'] / vars_merged['BuW_knn1_mean']
vars['HW_knn5_merged'] = vars['BuHt_wmean'] / vars_merged['BuW_knn5_mean']

vars['BuW_delaunay_mean_merged'] = vars_merged['BuW_delaunay_mean']
vars['BuW_knn5_mean_merged'] = vars_merged['BuW_knn5_mean']

vars['BuERI_mode_merged'] = vars_merged['BuERI_mode']
vars['BuERI_wmean_merged'] = vars_merged['BuERI_wmean']

temp = pd.read_csv('../../data/raw_data/ta_vp_rh_60_min_2022_09_01_2024_08_31_gap_filled_new.csv')
temp['datetime']=pd.to_datetime(temp['datetime'])
temp['datetime_UTC'] = temp['datetime']
temp['value'] = temp['ta']
temp = temp[temp['type'] == 'measured']

temp['datetime_UTC'] = temp['datetime_UTC'].astype(str)
temp = temp.pivot(index='station_id', columns='datetime_UTC', values='value')
temp.index = temp.index.str[2:]

In [6]:
hin = pd.read_csv('../../data/processed_data/2_year_heat_island_dates_night.csv')
hin = hin.values.flatten()

In [8]:
stats_df = pd.read_csv(f'../../data/processed_data/heat_island_stats_{radius}_.csv')

### Plots

In [9]:
radius = 300

In [10]:
stations = pd.read_csv("../../data/raw_data/Freiburg-Street-Level-Weather-Station-Network-MetaData-V1-0.csv")

In [ ]:
var_name_mapping = {
    'BuAre_sum': 'Plan Area Fraction ($\it{λ_P}$, m$^2$ m$^{-2}$)',
    'BuVol_3D_sum': 'Building Volume Density ($\it{λ_V}$, m$^3$ m$^{-2}$)',
    'BuEWA_3D_sum': 'Exposed Facet Density ($\it{λ_E}$, m$^2$ m$^{-2}$)', 
    'BuAdj': 'Adjacency ($\it{Adj}$)',
    'BuHt_wmean': 'Height ($\it{H}$, m)',
    'SVF_3D_mean': 'Sky View Factor ($\it{SVF}$)',
    'BuERI_mode_merged': 'Shape Complexity ($\it{SC}$)',
    'StrClo400_median': 'Street Network Connectivity ($\it{C}$)',
    'StrHW_wmean': 'Height:Width v3 ($\it{H:W_3}$)',
    'HW_geo':'Height:Width v2 ($\it{H:W_2}$)',
    'HW_del_merged':'Height:Width v1 ($\it{H:W_1}$)',
    'BuW_delaunay_mean_merged':'Width ($\it{W}$, m)',
    'BuW_delaunay_mean':'Width (cadastre) ($\it{W}$, m)',
    'HW_knn1_merged':'HW_knn1_merged',
    'HW_knn5_merged':'HW_knn5_merged',
    'NdDen':'$\it{ND}$'
    }

In [13]:
temp = temp.sub(temp.mean(axis=0), axis=1)
#temp = temp.div(temp.std(axis=0), axis=1)

vars_stn = vars.merge(temp, left_on='station_id', right_on='station_id',how='inner')

In [ ]:
reload(vis)

In [39]:
vars['BuAdj'] = 1 - vars['BuAdj']
vars['BuERI_mode_merged'] = 1 - vars['BuERI_mode_merged']

In [ ]:
reload(vis)

# make 3 by 4 plot with the following parameters
params = ['BuAre_sum', 'BuVol_3D_sum', 'BuEWA_3D_sum','BuHt_wmean', 'BuW_delaunay_mean_merged','SVF_3D_mean','HW_del_merged','HW_geo','StrHW_wmean', 'BuERI_mode_merged','BuAdj', 'StrClo400_median']

fig, axs = plt.subplots(4, 3, figsize=(18, 23))

for i, ax in enumerate(axs.flat):
    if params[i] in ['BuVol_3D_sum', 'BuEWA_3D_sum', 'SVF_3D_mean', 'HW_geo','StrHW_wmean']:
        vis.simple_plot_reduced(ax, 300, vars, params[i], hin, temp, stations, var_name_mapping, loess_frac=0.8)
    elif params[i] in ['BuHt_wmean', 'BuW_delaunay_mean_merged','StrClo400_median']:
        vis.simple_plot_reduced(ax, 300, vars, params[i], hin, temp, stations, var_name_mapping, loess_frac=0.95)
    else:
        vis.simple_plot_reduced(ax, 300, vars, params[i], hin, temp, stations, var_name_mapping, loess_frac = 0.85)
    ax.tick_params(axis='both', which='major', labelsize=14)
    ax.set_ylabel("$\it{ΔT_a}$ (K)", fontsize=16)
    #ax.set_ylim(-7,7)   
    ax.grid()

    if i % 3 != 0:  # 1st column (index 0, 3, 6, ...) keeps labels
        ax.set_ylabel("")  # Remove y-axis label
        ax.yaxis.set_ticklabels([])  # Remove y-axis ticks

    if i == 4:
        ax.set_xlim(10,250)

    if i == 11:
        ax.xaxis.set_major_formatter(ScalarFormatter())  # Set ScalarFormatter for x-axis
        ax.yaxis.set_major_formatter(ScalarFormatter())  # Set ScalarFormatter for y-axis
        ax.ticklabel_format(style='sci', scilimits=(0,0), axis='both')


legend_elements = vis.custom_lcz_legend()
ax.legend(ncol = 3, handles=legend_elements, loc='center', fontsize=14,bbox_to_anchor=(-0.9,-0.4))
#ax.set_axis_off()
  
plt.tight_layout(w_pad=0.01, h_pad=2)
fig.subplots_adjust(hspace=0.3)
plt.savefig('../../figures/fig5/fig_5_HW.png', dpi = 650)
plt.savefig('../../figures/fig5/fig_5_HW.pdf')
plt.savefig('../../figures/fig5/fig_5_HW.svg')
plt.show()

In [41]:
def calculate_slopes_loess(temp_lowess, turn_a, turn_b):
    
    # nearest LOESS point to breakpoint
    idx_a = np.argmin(np.abs(temp_lowess[:,0] - turn_a))
    idx_b = np.argmin(np.abs(temp_lowess[:,0] - turn_b))

    x_turn_a, y_turn_a = temp_lowess[idx_a]
    x_turn_b, y_turn_b = temp_lowess[idx_b]
    x_0, y_0 = temp_lowess[0]
    x_max, y_max = temp_lowess[-1]

    slope1 = (y_turn_a - y_0) / (x_turn_a - x_0)
    slope2 = (y_max - y_turn_b) / (x_max - x_turn_b)

    return slope1, slope2

In [ ]:
# LOESS Slopes
lowess = sm.nonparametric.lowess

fig, axs = plt.subplots(4, 3, figsize=(18, 23))

for i, ax in enumerate(axs.flat):
    if params[i] == 'BuAre_sum':
        data, mean, std, spearman_corr, p_value, pearson_corr, r_squared, rmse, cooks_d, mi, y_pred, slope, intercept, bse, summary = main.stats_multiple_times(vars, params[i], hin, temp)
        temp_lowess = lowess(endog=data['temperature'], exog=data[params[i]], is_sorted=False, frac=0.85)
        print(params[i], calculate_slopes_loess(temp_lowess, turn_a=0.15, turn_b=0.27))
    if params[i] == 'BuVol_3D_sum':
        data, mean, std, spearman_corr, p_value, pearson_corr, r_squared, rmse, cooks_d, mi, y_pred, slope, intercept, bse, summary = main.stats_multiple_times(vars, params[i], hin, temp)
        temp_lowess = lowess(endog=data['temperature'], exog=data[params[i]], is_sorted=False, frac=0.8)
        print(params[i], calculate_slopes_loess(temp_lowess, turn_a=1.9, turn_b=4.0))
    if params[i] == 'BuEWA_3D_sum':
        data, mean, std, spearman_corr, p_value, pearson_corr, r_squared, rmse, cooks_d, mi, y_pred, slope, intercept, bse, summary = main.stats_multiple_times(vars, params[i], hin, temp)
        temp_lowess = lowess(endog=data['temperature'], exog=data[params[i]], is_sorted=False, frac=0.8)
        print(params[i], calculate_slopes_loess(temp_lowess, turn_a=0.39, turn_b=0.55))
    if params[i] == 'BuHt_wmean':
        data, mean, std, spearman_corr, p_value, pearson_corr, r_squared, rmse, cooks_d, mi, y_pred, slope, intercept, bse, summary = main.stats_multiple_times(vars, params[i], hin, temp)
        temp_lowess = lowess(endog=data['temperature'], exog=data[params[i]], is_sorted=False, frac=0.95)
        print(params[i], calculate_slopes_loess(temp_lowess, turn_a=7.0, turn_b=8.5))    
    if params[i] == 'BuW_delaunay_mean_merged':
        data, mean, std, spearman_corr, p_value, pearson_corr, r_squared, rmse, cooks_d, mi, y_pred, slope, intercept, bse, summary = main.stats_multiple_times(vars, params[i], hin, temp)
        temp_lowess = lowess(endog=data['temperature'], exog=data[params[i]], is_sorted=False, frac=0.95)
        print(params[i], calculate_slopes_loess(temp_lowess, turn_a=80, turn_b=90))
    if params[i] == 'SVF_3D_mean':
        data, mean, std, spearman_corr, p_value, pearson_corr, r_squared, rmse, cooks_d, mi, y_pred, slope, intercept, bse, summary = main.stats_multiple_times(vars, params[i], hin, temp)
        temp_lowess = lowess(endog=data['temperature'], exog=data[params[i]], is_sorted=False, frac=0.8)
        print(params[i], calculate_slopes_loess(temp_lowess, turn_a=0.85, turn_b=0.95))
    if params[i] == 'HW_del_merged':
        data, mean, std, spearman_corr, p_value, pearson_corr, r_squared, rmse, cooks_d, mi, y_pred, slope, intercept, bse, summary = main.stats_multiple_times(vars, params[i], hin, temp)
        temp_lowess = lowess(endog=data['temperature'], exog=data[params[i]], is_sorted=False, frac=0.85)
        print(params[i], calculate_slopes_loess(temp_lowess, turn_a=0.5, turn_b=0.6))   
    if params[i] == 'HW_geo':
        data, mean, std, spearman_corr, p_value, pearson_corr, r_squared, rmse, cooks_d, mi, y_pred, slope, intercept, bse, summary = main.stats_multiple_times(vars, params[i], hin, temp)
        temp_lowess = lowess(endog=data['temperature'], exog=data[params[i]], is_sorted=False, frac=0.8)
        print(params[i], calculate_slopes_loess(temp_lowess, turn_a=0.5, turn_b=1))
    if params[i] == 'StrHW_wmean':
        data, mean, std, spearman_corr, p_value, pearson_corr, r_squared, rmse, cooks_d, mi, y_pred, slope, intercept, bse, summary = main.stats_multiple_times(vars, params[i], hin, temp)
        temp_lowess = lowess(endog=data['temperature'], exog=data[params[i]], is_sorted=False, frac=0.8)
        print(params[i], calculate_slopes_loess(temp_lowess, turn_a=0.3, turn_b=0.5))
    if params[i] == 'BuERI_mode_merged':
        data, mean, std, spearman_corr, p_value, pearson_corr, r_squared, rmse, cooks_d, mi, y_pred, slope, intercept, bse, summary = main.stats_multiple_times(vars, params[i], hin, temp)
        temp_lowess = lowess(endog=data['temperature'], exog=data[params[i]], is_sorted=False, frac=0.85)
        print(params[i], calculate_slopes_loess(temp_lowess, turn_a=0.45, turn_b=0.6))
    if params[i] == 'BuAdj':
        data, mean, std, spearman_corr, p_value, pearson_corr, r_squared, rmse, cooks_d, mi, y_pred, slope, intercept, bse, summary = main.stats_multiple_times(vars, params[i], hin, temp)
        temp_lowess = lowess(endog=data['temperature'], exog=data[params[i]], is_sorted=False, frac=0.85)
        print(params[i], calculate_slopes_loess(temp_lowess, turn_a=0.4, turn_b=0.5))
    if params[i] == 'StrClo400_median':
        data, mean, std, spearman_corr, p_value, pearson_corr, r_squared, rmse, cooks_d, mi, y_pred, slope, intercept, bse, summary = main.stats_multiple_times(vars, params[i], hin, temp)
        temp_lowess = lowess(endog=data['temperature'], exog=data[params[i]], is_sorted=False, frac=0.95)
        print(params[i], calculate_slopes_loess(temp_lowess, turn_a=0.00001, turn_b=0.000012))



# Winter

In [ ]:

# make 3 by 4 plot with the following parameters
params = ['BuAre_sum', 'BuVol_3D_sum', 'BuEWA_3D_sum','BuHt_wmean', 'BuW_delaunay_mean_merged','SVF_3D_mean','HW_del_merged','HW_geo','StrHW_wmean', 'BuERI_mode_merged','BuAdj', 'StrClo400_median']

fig, axs = plt.subplots(4, 3, figsize=(18, 23))

for i, ax in enumerate(axs.flat):
    vis.simple_plot_reduced(ax, 300, vars, params[i], hiwn, temp, stations, var_name_mapping)
    ax.tick_params(axis='both', which='major', labelsize=14)
    ax.set_ylabel("$\it{ΔT_a}$ (Air Temperature  \n Relative to Daily Mean, K)", fontsize=16)
    #ax.set_ylim(-7,7)   
    ax.grid()

    if i % 3 != 0:  # 1st column (index 0, 3, 6, ...) keeps labels
        ax.set_ylabel("")  # Remove y-axis label
        ax.yaxis.set_ticklabels([])  # Remove y-axis ticks

    if i == 4:
        ax.set_xlim(10,250)

    if i == 11:
        ax.xaxis.set_major_formatter(ScalarFormatter())  # Set ScalarFormatter for x-axis
        ax.yaxis.set_major_formatter(ScalarFormatter())  # Set ScalarFormatter for y-axis
        ax.ticklabel_format(style='sci', scilimits=(0,0), axis='both')


legend_elements = vis.custom_lcz_legend()
ax.legend(ncol = 3, handles=legend_elements, loc='center', fontsize=14,bbox_to_anchor=(-0.9,-0.4))
#ax.set_axis_off()
  
plt.tight_layout(w_pad=0.01, h_pad=2)
fig.subplots_adjust(hspace=0.3)
plt.savefig('../../figures/fig5/fig_5_winter.png', dpi = 650)
plt.savefig('../../figures/fig5/fig_5_winter.pdf')
plt.savefig('../../figures/fig5/fig_5_winter.svg')
plt.show()

# Summer

In [ ]:

# make 3 by 4 plot with the following parameters
params = ['BuAre_sum', 'BuVol_3D_sum', 'BuEWA_3D_sum','BuHt_wmean', 'BuW_delaunay_mean_merged','SVF_3D_mean','HW_del_merged','HW_geo','StrHW_wmean', 'BuERI_mode_merged','BuAdj', 'StrClo400_median']

fig, axs = plt.subplots(4, 3, figsize=(18, 23))

for i, ax in enumerate(axs.flat):
    vis.simple_plot_reduced(ax, 300, vars, params[i], hisn, temp, stations, var_name_mapping)
    ax.tick_params(axis='both', which='major', labelsize=14)
    ax.set_ylabel("$\it{ΔT_a}$ (Air Temperature  \n Relative to Daily Mean, K)", fontsize=16)
    #ax.set_ylim(-7,7)   
    ax.grid()

    if i % 3 != 0:  # 1st column (index 0, 3, 6, ...) keeps labels
        ax.set_ylabel("")  # Remove y-axis label
        ax.yaxis.set_ticklabels([])  # Remove y-axis ticks

    if i == 4:
        ax.set_xlim(10,250)

    if i == 11:
        ax.xaxis.set_major_formatter(ScalarFormatter())  # Set ScalarFormatter for x-axis
        ax.yaxis.set_major_formatter(ScalarFormatter())  # Set ScalarFormatter for y-axis
        ax.ticklabel_format(style='sci', scilimits=(0,0), axis='both')


legend_elements = vis.custom_lcz_legend()
ax.legend(ncol = 3, handles=legend_elements, loc='center', fontsize=14,bbox_to_anchor=(-0.9,-0.4))
#ax.set_axis_off()
  
plt.tight_layout(w_pad=0.01, h_pad=2)
fig.subplots_adjust(hspace=0.3)
plt.savefig('../../figures/fig5/fig_5_summer.png', dpi = 650)
plt.savefig('../../figures/fig5/fig_5_summer.pdf')
plt.savefig('../../figures/fig5/fig_5_summer.svg')
plt.show()

# Spring

In [ ]:

# make 3 by 4 plot with the following parameters
params = ['BuAre_sum', 'BuVol_3D_sum', 'BuEWA_3D_sum','BuHt_wmean', 'BuW_delaunay_mean_merged','SVF_3D_mean','HW_del_merged','HW_geo','StrHW_wmean', 'BuERI_mode_merged','BuAdj', 'StrClo400_median']

fig, axs = plt.subplots(4, 3, figsize=(18, 23))

for i, ax in enumerate(axs.flat):
    vis.simple_plot_reduced(ax, 300, vars, params[i], hispn, temp, stations, var_name_mapping)
    ax.tick_params(axis='both', which='major', labelsize=14)
    ax.set_ylabel("$\it{ΔT_a}$ (Air Temperature  \n Relative to Daily Mean, K)", fontsize=16)
    #ax.set_ylim(-7,7)   
    ax.grid()

    if i % 3 != 0:  # 1st column (index 0, 3, 6, ...) keeps labels
        ax.set_ylabel("")  # Remove y-axis label
        ax.yaxis.set_ticklabels([])  # Remove y-axis ticks

    if i == 4:
        ax.set_xlim(10,250)

    if i == 11:
        ax.xaxis.set_major_formatter(ScalarFormatter())  # Set ScalarFormatter for x-axis
        ax.yaxis.set_major_formatter(ScalarFormatter())  # Set ScalarFormatter for y-axis
        ax.ticklabel_format(style='sci', scilimits=(0,0), axis='both')


legend_elements = vis.custom_lcz_legend()
ax.legend(ncol = 3, handles=legend_elements, loc='center', fontsize=14,bbox_to_anchor=(-0.9,-0.4))
#ax.set_axis_off()
  
plt.tight_layout(w_pad=0.01, h_pad=2)
fig.subplots_adjust(hspace=0.3)
plt.savefig('../../figures/fig5/fig_5_spring.png', dpi = 650)
plt.savefig('../../figures/fig5/fig_5_spring.pdf')
plt.savefig('../../figures/fig5/fig_5_spring.svg')
plt.show()

# Autumns

In [ ]:

# make 3 by 4 plot with the following parameters
params = ['BuAre_sum', 'BuVol_3D_sum', 'BuEWA_3D_sum','BuHt_wmean', 'BuW_delaunay_mean_merged','SVF_3D_mean','HW_del_merged','HW_geo','StrHW_wmean', 'BuERI_mode_merged','BuAdj', 'StrClo400_median']

fig, axs = plt.subplots(4, 3, figsize=(18, 23))

for i, ax in enumerate(axs.flat):
    vis.simple_plot_reduced(ax, 300, vars, params[i], hian, temp, stations, var_name_mapping)
    ax.tick_params(axis='both', which='major', labelsize=14)
    ax.set_ylabel("$\it{ΔT_a}$ (Air Temperature  \n Relative to Daily Mean, K)", fontsize=16)
    #ax.set_ylim(-7,7)   
    ax.grid()

    if i % 3 != 0:  # 1st column (index 0, 3, 6, ...) keeps labels
        ax.set_ylabel("")  # Remove y-axis label
        ax.yaxis.set_ticklabels([])  # Remove y-axis ticks

    if i == 4:
        ax.set_xlim(10,250)

    if i == 11:
        ax.xaxis.set_major_formatter(ScalarFormatter())  # Set ScalarFormatter for x-axis
        ax.yaxis.set_major_formatter(ScalarFormatter())  # Set ScalarFormatter for y-axis
        ax.ticklabel_format(style='sci', scilimits=(0,0), axis='both')


legend_elements = vis.custom_lcz_legend()
ax.legend(ncol = 3, handles=legend_elements, loc='center', fontsize=14,bbox_to_anchor=(-0.9,-0.4))
#ax.set_axis_off()
  
plt.tight_layout(w_pad=0.01, h_pad=2)
fig.subplots_adjust(hspace=0.3)
plt.savefig('../../figures/fig5/fig_5_autumn.png', dpi = 650)
plt.savefig('../../figures/fig5/fig_5_autumn.pdf')
plt.savefig('../../figures/fig5/fig_5_autumn.svg')
plt.show()

# All year

In [ ]:

# make 3 by 4 plot with the following parameters
params = ['BuAre_sum', 'BuVol_3D_sum', 'BuEWA_3D_sum','BuHt_wmean', 'BuW_delaunay_mean_merged','SVF_3D_mean','HW_del_merged','HW_geo','StrHW_wmean', 'BuERI_mode_merged','BuAdj', 'StrClo400_median']

fig, axs = plt.subplots(4, 3, figsize=(18, 23))

for i, ax in enumerate(axs.flat):
    vis.simple_plot_reduced(ax, 300, vars, params[i], temp.columns.values, temp, stations, var_name_mapping)
    ax.tick_params(axis='both', which='major', labelsize=14)
    ax.set_ylabel("$\it{ΔT_a}$ (Air Temperature  \n Relative to Daily Mean, K)", fontsize=16)
    #ax.set_ylim(-7,7)   
    ax.grid()

    if i % 3 != 0:  # 1st column (index 0, 3, 6, ...) keeps labels
        ax.set_ylabel("")  # Remove y-axis label
        ax.yaxis.set_ticklabels([])  # Remove y-axis ticks

    if i == 4:
        ax.set_xlim(10,250)

    if i == 11:
        ax.xaxis.set_major_formatter(ScalarFormatter())  # Set ScalarFormatter for x-axis
        ax.yaxis.set_major_formatter(ScalarFormatter())  # Set ScalarFormatter for y-axis
        ax.ticklabel_format(style='sci', scilimits=(0,0), axis='both')


legend_elements = vis.custom_lcz_legend()
ax.legend(ncol = 3, handles=legend_elements, loc='center', fontsize=14,bbox_to_anchor=(-0.9,-0.4))
#ax.set_axis_off()
  
plt.tight_layout(w_pad=0.01, h_pad=2)
fig.subplots_adjust(hspace=0.3)
plt.savefig('../../figures/fig5/fig_5_all_year.png', dpi = 650)
plt.savefig('../../figures/fig5/fig_5_all_year.pdf')
plt.savefig('../../figures/fig5/fig_5_all_year.svg')
plt.show()